In [1]:
!pip install transformers accelerate sentence-transformers faiss-cpu langchain-text-splitters -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 18.1 MB/s eta 0:00:00


In [2]:
with open("knowledge.txt") as f:
    knowledge_text = f.read()

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

with open("knowledge.txt") as f:
    knowledge_text = f.read()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_text(knowledge_text)

print(f"We have {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---\n{chunk}\n")

We have 3 chunks:
--- Chunk 1 ---
Manual interno da empresa

Trabalho remoto: os funcionários podem trabalhar remotamente às segundas e sextas-feiras. Às terças, quartas e quintas-feiras, o trabalho deve ser presencial.

--- Chunk 2 ---
Férias: funcionários em período integral recebem 20 dias de férias por ano. O saldo é acumulado mensalmente.

Tecnologia: o backend oficial utiliza Python. O frontend utiliza React. Aplicações móveis utilizam React Native.

--- Chunk 3 ---
Suporte: solicitações de suporte devem ser abertas no sistema interno durante o horário comercial, de segunda a sexta-feira, das 08h às 18h.



In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_embeddings = embedding_model.encode(chunks)

print(chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(3, 384)


In [5]:
import faiss
import numpy as np

d = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(d)

index.add(
    np.array(chunk_embeddings).astype("float32")
)

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 3 vectors.


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llm_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [8]:
def generate_answer(context, query):
    messages = [
        {
            "role": "system",
            "content": (
                "Você responde perguntas usando somente o contexto fornecido. "
                "Se a resposta não estiver no contexto, diga que não há "
                "informação suficiente."
            )
        },
        {
            "role": "user",
            "content": f"""
Contexto:
{context}

Pergunta:
{query}

Responda em português, de forma curta e objetiva.
"""
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(llm_model.device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [9]:
def answer_question(query):
    # RETRIEVE
    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        k=1
    )

    retrieved_chunks = [
        chunks[i]
        for i in indices[0]
    ]

    context = "\n\n".join(retrieved_chunks)

    # GENERATE
    answer = generate_answer(context, query)

    print("--- CONTEXT ---")
    print(context)
    print("-----------------")

    return answer

In [11]:
query_1 = "Quantos são os dias de férias?"

print(f"Query: {query_1}")
print(f"Answer: {answer_question(query_1)}")

Query: Quantos são os dias de férias?
--- CONTEXT ---
Férias: funcionários em período integral recebem 20 dias de férias por ano. O saldo é acumulado mensalmente.

Tecnologia: o backend oficial utiliza Python. O frontend utiliza React. Aplicações móveis utilizam React Native.
-----------------
Answer: 20 dias de férias por ano.
